# AI-Generated Image Detector

Upload an image, get a calibrated probability that it is AI-generated.

A two-branch detector: a **frozen CLIP embedding** for semantics plus
**hand-designed frequency features** for forensics, mixed by a *degradation-aware
gate* that estimates how damaged the image is and weights the branches
accordingly. The forensic branch is precise on clean images and collapses under
compression; the semantic branch is coarser but survives. Only the small head is
trained — 563,724 parameters on top of a frozen 86M backbone.

Trained on [SID_Set](https://huggingface.co/datasets/saberzl/SID_Set) with a
targeted hard-negative pass. Held-out accuracy **0.996**, F1 **0.993**,
ROC-AUC **1.000** — and no transform cell (JPEG q30, blur σ=2, 0.25× downscale,
noise σ=0.1, colour jitter, crop) drops AUC below **0.999**.

Runs on **CPU** at ~8 images/s, so no GPU runtime is needed.
`Runtime → Run all`, then upload an image in section 4.

## 1. Install dependencies

In [ ]:
# torch/torchvision ship with Colab; these are the rest.
!pip install -q open_clip_torch scipy scikit-learn Pillow

import torch
print('torch', torch.__version__, '|', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. Get the code and the trained weights

The checkpoint (`checkpoints/full.pt`, 2.2 MB) is committed in the repo, so a
single clone brings both.

In [ ]:
import os, sys, pathlib, subprocess

REPO = 'https://github.com/mxhxmza/aigc-detector.git'
PROJECT = pathlib.Path('/content/aigc-detector/aigc-detector')

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO,
                    '/content/aigc-detector'], check=True)

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
CKPT = PROJECT / 'checkpoints' / 'full.pt'
print('code   :', PROJECT)
print('weights:', CKPT, f'({CKPT.stat().st_size // 1024} KB)' if CKPT.exists() else 'MISSING')

## 3. Load the model

The first run downloads the frozen CLIP ViT-B/16 backbone (~350 MB) through
`open_clip`; it is cached for the rest of the session.

In [ ]:
from src.inference import Scorer

scorer = Scorer(CKPT, device='auto')
p = scorer.params
print(f"device      : {scorer.device}")
print(f"backbone    : {scorer.config['backbone']}")
print(f"temperature : {scorer.temperature:.3f}")
print(f"parameters  : {p['total']:,} total "
      f"({p['trainable']:,} trainable + {p['frozen_backbone']:,} frozen)")

## 4. Score your own images

Run the cell, pick one or more files. Each result shows the calibrated
probability and the same verdict band the web app uses.

In [ ]:
from PIL import Image
from IPython.display import display

def verdict(p):
    if p >= 0.85: return 'Very likely AI-generated'
    if p >= 0.60: return 'Probably AI-generated'
    if p >  0.40: return 'Uncertain'
    if p >  0.15: return 'Probably authentic'
    return 'Very likely authentic'

def score(paths, preview=True):
    paths = [str(x) for x in paths]
    imgs = [Image.open(x).convert('RGB') for x in paths]
    for path, img, p in zip(paths, imgs, scorer.score_many(imgs)):
        if preview:
            thumb = img.copy(); thumb.thumbnail((260, 260)); display(thumb)
        print(f"  {os.path.basename(path)}")
        print(f"  p(AI-generated) = {p:.1%}   ->   {verdict(p)}\n")

from google.colab import files
score(files.upload().keys())

## 5. Optional — score a folder to JSON

The graded CLI: walks a directory and writes
`[{"image_path": ..., "pred": ...}, ...]`. Uncomment to use.

In [ ]:
# !python predict.py --image-dir /content/my_images --out /content/predictions.json
# !head -20 /content/predictions.json

---

### Reading the score

`pred` is a **calibrated** probability in [0, 1] that the image is fully
AI-generated; **0.5** is the classification threshold. Temperature scaling was
fitted on held-out data after model selection, so the number is meant to be read
at face value rather than as a bare logit.

A *tampered* image — a real photograph with an AI-edited region — is treated as
**real**, because it was still taken by a person. Only fully synthetic images are
flagged.

### Held-out results (2,255 images never trained on)

| metric | value |
|---|---|
| Accuracy | 0.996 |
| Precision | 0.991 |
| Recall | 0.996 |
| F1 | 0.993 |
| ROC-AUC | 1.000 |

7 false positives (0.5% of real photos), 3 false negatives (0.4% of AI).

### Known limitation

Training reals skew casual, so unusually *polished* photographs — heavy
foreground bokeh, saturated travel framing — were the model's weak spot. A
hard-negative pass (`scripts/add_hard_negatives.py`) cut that error rate from
1.2% to 0.4%, but photographs far outside the training distribution can still
be misjudged. Calling a real photo synthetic has a human cost, so the
appropriate deployment is a human-review queue, not automated enforcement.

Source: [github.com/mxhxmza/aigc-detector](https://github.com/mxhxmza/aigc-detector) · MIT